<a href="https://colab.research.google.com/github/Brandeis-Visual-Analytics/COSI-165b-labs/blob/main/lab5/lab_5_Overspecified_Models_Double_Descent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5: Overspecifying Models with Large Numbers of Parameters

**REMEMBER TO File > Save a Copy in Drive IF YOU ARE USING COLAB**

**BEFORE YOU RUN ANYTHING, turn on a GPU:** in Colab, choose **Runtime > Change runtime type**, select **T4 GPU** as the hardware accelerator, and click Save. (Changing the runtime restarts your session, so do this first.) Part 2 trains several networks and is much faster on a GPU. If you are not using Colab, the code will fall back to the CPU and be slower.

This notebook has two parts:

* **Part 1: MNIST-1D Performance.** You'll define and train a small fully connected network on the MNIST-1D dataset (figure 8.2a of the book), look at its training and test performance, and think about how trustworthy your test results really are.
* **Part 2: Double Descent.** You'll reuse the model you defined in Part 1 to train networks with more and more hidden units, all the way up to models with far more parameters than there are training examples, and interpret the resulting double descent curve, as described in section 8.4 of the book.

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Adapted from notebooks 8_1 and 8_3 for *Understanding Deep Learning* by Simon J.D. Prince (udlbookmail@gmail.com).

In [ ]:
# Run this if you're in a Colab to install the MNIST-1D repository (used in both parts)
%pip install git+https://github.com/greydanus/mnist1d

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import StepLR
import numpy as np
import matplotlib.pyplot as plt
import mnist1d
import random
random.seed(0)

# Use the GPU if there is one (in Colab: Runtime > Change runtime type > T4 GPU)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using:', DEVICE)

# **Part 1: MNIST-1D Performance**

This part runs a simple neural network on the MNIST-1D dataset as in figure 8.2a. MNIST-1D is a small synthetic stand-in for MNIST: each example is a one-dimensional signal of length 40 that belongs to one of 10 classes (a noisy, shifted, scaled template for each of the digits 0 to 9). It uses code from https://github.com/greydanus/mnist1d to generate the data.

Let's generate a training and test dataset using the MNIST1D code.  The dataset gets saved as a .pkl file so it doesn't have to be regenerated each time.

In [ ]:
args = mnist1d.data.get_dataset_args()
data = mnist1d.data.get_dataset(args, path='./mnist1d_data.pkl', download=False, regenerate=False)

# The training and test input and outputs are in
# data['x'], data['y'], data['x_test'], and data['y_test']
print("Examples in training set: {}".format(len(data['y'])))
print("Examples in test set: {}".format(len(data['y_test'])))
print("Length of each example: {}".format(data['x'].shape[-1]))

Now we'll define the model. We'll write it as a function `get_model(n_hidden)` that returns a freshly initialized network with two hidden layers of `n_hidden` units each, so that we can reuse it in Part 2 with different sizes. There are two things to write:

1. `weights_init()`, which applies He initialization to the weights of each linear layer (and sets the biases to zero).
2. `get_model()`, which builds the network with two hidden layers and ReLU activations between them.

We also give you `count_parameters()`, which we'll use to compare the size of the model to the size of the training set.

In [ ]:
D_i = 40    # Input dimensions
D_o = 10    # Output dimensions

def weights_init(layer_in):
  # TODO:
  # Initialize the weights of each linear layer with He initialization, and set the biases to zero.
  # Hint: PyTorch has He initialization built in. See torch.nn.init.kaiming_uniform_:
  #   https://docs.pytorch.org/docs/2.14/nn.init.html#torch.nn.init.kaiming_uniform_
  # Note that model.apply() calls this function on every module in the model, including the ReLUs,
  # so check that layer_in is an nn.Linear before changing its parameters.
  # Replace this line (see figure 7.8 of book for help)
  print("Initializing layer")


# Return an initialized model with two hidden layers and n_hidden hidden units in each
def get_model(n_hidden):
  D_k = n_hidden   # Hidden dimensions

  # TODO:
  # Define a model with two hidden layers of size D_k
  # And ReLU activations between them
  # Replace this line (see Figure 7.8 of book for help):
  model = nn.Sequential(nn.Linear(D_i, D_o))

  # Initialize the parameters using the function you just defined
  model.apply(weights_init)

  return model

def count_parameters(model):
  return sum(p.numel() for p in model.parameters() if p.requires_grad)

# The model for this part has two hidden layers of 100 units
D_k = 100
model = get_model(D_k)
print(model)
print(f"Number of parameters in the model: {count_parameters(model)}")
print(f"Number of training examples: {len(data['y'])}")

In [ ]:
# choose cross entropy loss function (equation 5.24)
loss_function = torch.nn.CrossEntropyLoss()
# construct SGD optimizer and initialize learning rate and momentum
optimizer = torch.optim.SGD(model.parameters(), lr = 0.05, momentum=0.9)
# object that decreases learning rate by half every 10 epochs
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)
x_train = torch.tensor(data['x'].astype('float32'))
y_train = torch.tensor(data['y'].transpose().astype('int64'))
x_test= torch.tensor(data['x_test'].astype('float32'))
y_test = torch.tensor(data['y_test'].astype('int64'))

# load the data into a class that creates the batches
data_loader = DataLoader(TensorDataset(x_train,y_train), batch_size=100, shuffle=True, worker_init_fn=np.random.seed(1))

# Initialize model weights
model.apply(weights_init)

# loop over the dataset n_epoch times
n_epoch = 50
# store the loss and the % correct at each epoch
losses_train = np.zeros((n_epoch))
errors_train = np.zeros((n_epoch))
losses_test = np.zeros((n_epoch))
errors_test = np.zeros((n_epoch))

for epoch in range(n_epoch):
  # loop over batches
  for i, batch in enumerate(data_loader):
    # retrieve inputs and labels for this batch
    x_batch, y_batch = batch
    # zero the parameter gradients
    optimizer.zero_grad()
    # forward pass -- calculate model output
    pred = model(x_batch)
    # compute the loss
    loss = loss_function(pred, y_batch)
    # backward pass
    loss.backward()
    # SGD update
    optimizer.step()

  # Run whole dataset to get statistics -- normally wouldn't do this
  pred_train = model(x_train)
  pred_test = model(x_test)
  _, predicted_train_class = torch.max(pred_train.data, 1)
  _, predicted_test_class = torch.max(pred_test.data, 1)
  errors_train[epoch] = 100 - 100 * (predicted_train_class == y_train).float().sum() / len(y_train)
  errors_test[epoch]= 100 - 100 * (predicted_test_class == y_test).float().sum() / len(y_test)
  losses_train[epoch] = loss_function(pred_train, y_train).item()
  losses_test[epoch]= loss_function(pred_test, y_test).item()
  print(f'Epoch {epoch:5d}, train loss {losses_train[epoch]:.6f}, train error {errors_train[epoch]:3.2f},  test loss {losses_test[epoch]:.6f}, test error {errors_test[epoch]:3.2f}')

  # tell scheduler to consider updating learning rate
  scheduler.step()

In [ ]:
# Plot the results
fig, ax = plt.subplots()
ax.plot(errors_train,'r-',label='train')
ax.plot(errors_test,'b-',label='test')
ax.set_ylim(0,100); ax.set_xlim(0,n_epoch)
ax.set_xlabel('Epoch'); ax.set_ylabel('Error')
ax.set_title('TrainError %3.2f, Test Error %3.2f'%(errors_train[-1],errors_test[-1]))
ax.legend()
plt.show()

# Plot the results
fig, ax = plt.subplots()
ax.plot(losses_train,'r-',label='train')
ax.plot(losses_test,'b-',label='test')
ax.set_xlim(0,n_epoch)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Train loss %3.2f, Test loss %3.2f'%(losses_train[-1],losses_test[-1]))
ax.legend()
plt.show()

**TODO**

Play with the model -- try changing the number of layers, hidden units, learning rate, batch size, momentum or anything else you like.  See if you can improve the test results.

Is it a good idea to optimize the hyperparameters in this way?  Will the final result be a good estimate of the true test performance?

Also take a look at the size of this model. With two hidden layers of 100 units it should have about 15,000 parameters (check the count printed above), but the training set has only 4,000 examples. Compare the training error and the test error in your plots. What does the gap between them tell you? In Part 2 we'll explore what happens as we change the number of parameters relative to the number of training examples.

# **Part 2: Double Descent**

In Part 1 your network had many more parameters than training examples. We call a model like this **overspecified** (or overparameterized): it has more than enough capacity to fit the training data exactly. Classical intuition says this should lead to terrible overfitting. Section 8.4 of the book describes why that is only half of the story. As the number of parameters grows past the point where the model can just barely fit the training data perfectly, the test error can go *down* a second time. This is called **double descent**.

We'll reuse `weights_init()`, `get_model()` and `count_parameters()` from Part 1, this time calling `get_model()` with many different numbers of hidden units. **Before going on, make sure those functions work**: the model in Part 1 should have 15,210 parameters. If it doesn't, fix your code first, because everything below depends on it.

To see the effect clearly we'll use a modified version of MNIST-1D, with 4,000 training examples and 4,000 test examples, extra noise in the data, and **15% of the training labels replaced with random classes**. The noisy labels make the training set harder to fit, so the network needs more capacity before it can memorize it all, and the effect is easier to see.

In [ ]:
# Note: we save this dataset to a different file than in Part 1 so that we don't overwrite it
args = mnist1d.data.get_dataset_args()
args.num_samples = 8000
args.train_split = 0.5
args.corr_noise_scale = 0.25
args.iid_noise_scale=2e-2
data = mnist1d.data.get_dataset(args, path='./mnist1d_data_double_descent.pkl', download=False, regenerate=True)

# Add 15% noise to training labels
for c_y in range(len(data['y'])):
    random_number = random.random()
    if random_number < 0.15 :
        random_int = int(random.random() * 10)
        data['y'][c_y] = random_int

# The training and test input and outputs are in
# data['x'], data['y'], data['x_test'], and data['y_test']
print("Examples in training set: {}".format(len(data['y'])))
print("Examples in test set: {}".format(len(data['y_test'])))
print("Dimensionality of each example: {}".format(data['x'].shape[-1]))

Next is the function that trains a model and returns its final training and test errors. It's the same kind of training loop you saw in Part 1, except that it puts the model and the data on the GPU (if there is one) and forms the batches directly on the device, which is much faster than copying every batch from the CPU.

In [ ]:
def fit_model(model, data, n_epoch):

  # Move the model to the GPU (if there is one)
  model = model.to(DEVICE)

  # choose cross entropy loss function (equation 5.24)
  loss_function = torch.nn.CrossEntropyLoss()
  # construct SGD optimizer and initialize learning rate and momentum
  optimizer = torch.optim.SGD(model.parameters(), lr = 0.01, momentum=0.9)

  # put all of the data on the device once, so we don't copy batches back and forth
  x_train = torch.tensor(data['x'].astype('float32')).to(DEVICE)
  y_train = torch.tensor(data['y'].transpose().astype('long')).to(DEVICE)
  x_test = torch.tensor(data['x_test'].astype('float32')).to(DEVICE)
  y_test = torch.tensor(data['y_test'].astype('long')).to(DEVICE)

  n_train = x_train.shape[0]
  batch_size = 100

  for epoch in range(n_epoch):
    # shuffle the training data and loop over batches
    perm = torch.randperm(n_train, device=DEVICE)
    for start in range(0, n_train, batch_size):
      idx = perm[start:start+batch_size]
      # retrieve inputs and labels for this batch
      x_batch, y_batch = x_train[idx], y_train[idx]
      # zero the parameter gradients
      optimizer.zero_grad()
      # forward pass -- calculate model output
      pred = model(x_batch)
      # compute the loss
      loss = loss_function(pred, y_batch)
      # backward pass
      loss.backward()
      # SGD update
      optimizer.step()

    # Run whole dataset to get statistics (only when we print them, and after the last epoch)
    if epoch % 100 == 0 or epoch == n_epoch - 1:
      with torch.no_grad():
        pred_train = model(x_train)
        pred_test = model(x_test)
        _, predicted_train_class = torch.max(pred_train, 1)
        _, predicted_test_class = torch.max(pred_test, 1)
        errors_train = 100 - 100 * (predicted_train_class == y_train).float().mean().item()
        errors_test = 100 - 100 * (predicted_test_class == y_test).float().mean().item()
        losses_train = loss_function(pred_train, y_train).item()
        losses_test = loss_function(pred_test, y_test).item()
      if epoch % 100 == 0:
        print(f'Epoch {epoch:5d}, train loss {losses_train:.6f}, train error {errors_train:3.2f},  test loss {losses_test:.6f}, test error {errors_test:3.2f}')

  return errors_train, errors_test

The following code produces the double descent curve by training the model with different numbers of hidden units and plotting the test error.

**TODO**


*Before* you run the code, and considering that there are 4000 training examples and that you are going to try six different model sizes predict:<br>

1.    At what capacity do you think the training error will become zero?
2.   At what capacity do you expect the first minima of the double descent curve to appear?
3. At what capacity do you expect the maximum of the double descent curve to appear?

In [ ]:
# This code should take about 6 minutes to run on a GPU.  Check your email, help a neighbor, have a zen moment.

# Number of hidden units in each of the two hidden layers, for each model we'll train.
# (The book's figure uses many more sizes.  If you want the full, smoother curve, use the longer list
# below after class: it has 28 sizes and takes about 30 minutes.)
hidden_variables = np.array([6, 10, 16, 22, 28, 40, 70, 200])
# hidden_variables = np.array([2,4,6,8,10,14,18,22,26,30,35,40,45,50,55,60,70,80,90,100,120,140,160,180,200,250,300,400])

errors_train_all = np.zeros(len(hidden_variables))
errors_test_all = np.zeros(len(hidden_variables))
total_weights_all = np.zeros(len(hidden_variables), dtype=int)

# loop over the dataset n_epoch times
n_epoch = 1000

# For each hidden variable size
for c_hidden in range(len(hidden_variables)):
    print(f'Training model with {hidden_variables[c_hidden]:3d} hidden variables')
    # Get a model (using the function you wrote in Part 1)
    model = get_model(int(hidden_variables[c_hidden]))
    # Count and store number of weights
    total_weights_all[c_hidden] = count_parameters(model)
    # Train the model
    errors_train, errors_test = fit_model(model, data, n_epoch)
    # Store the results
    errors_train_all[c_hidden] = errors_train
    errors_test_all[c_hidden] = errors_test

Now plot the results. The dashed green line marks the model whose number of parameters is closest to the number of training examples.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Assuming data['y'] is available and contains the training examples
num_training_examples = len(data['y'])

# Find the index where total_weights_all is closest to num_training_examples
closest_index = np.argmin(np.abs(np.array(total_weights_all) - num_training_examples))

# Get the corresponding value of hidden variables
hidden_variable_at_num_training_examples = hidden_variables[closest_index]

# Plot the results
fig, ax = plt.subplots()
ax.plot(hidden_variables, errors_train_all, 'r-', label='train')
ax.plot(hidden_variables, errors_test_all, 'b-', label='test')

# Add a vertical line at the point where total weights equal the number of training examples
ax.axvline(x=hidden_variable_at_num_training_examples, color='g', linestyle='--', label='N(weights) = N(train)')

ax.set_ylim(0, 100)
ax.set_xlabel('No. hidden variables')
ax.set_ylabel('Error')
ax.legend()
plt.show()

**TODO**

Now interpret your double descent curve. Write your answers in a text cell below, or bring them to lab.

1. **Compare to your predictions.** Looking at the models you trained, at what number of hidden units did the training error first reach zero? Which model had the lowest test error before the peak, and which had the highest test error overall? How close were your predictions, and how do these locations compare to the dashed green line?
2. **Describe the three regimes.** Describe what happens to the training error and test error when the model has (a) too few parameters to fit the training data, (b) roughly as many parameters as training examples, and (c) many more parameters than training examples.
3. **Why does the test error peak?** Near the dashed line, the model has just enough capacity to fit the training data, including the 15% of labels that are wrong. What does it have to do to fit them, and why might that hurt on the test set?
4. **Why does the test error come down again?** Past the peak there are many different settings of the parameters that fit the training data perfectly, and the model has lots of spare capacity. Section 8.4 of the book suggests why the solution that training finds might still generalize well. Summarize its explanation in your own words.
5. **Connect back to Part 1.** The network in Part 1 had about 15,000 parameters for 4,000 training examples. Where would it sit on this curve? Did its test error look like the error at the peak, or like the error out on the right?
6. **Optional experiments.** We only trained eight model sizes to save time, so the curve is coarse and the exact location of the peak is uncertain. Try adding sizes near the dashed line (for example 35 and 45) to locate the peak more precisely. What would you expect to happen to the curve if we removed the label noise (change 0.15 in the data cell to 0 and re-run)? What if we trained for fewer epochs, or added more training data?